# Phase 6 — Match and group simulation

This self-contained notebook turns the Phase 5 score distributions into seeded match and four-team round-robin simulations. Standings use points, goal difference, goals scored, then team name as an explicit deterministic fallback. The alphabetical fallback is not a competition-specific sporting tie-break; it makes unresolved simulated ties reproducible.

In [1]:
from dataclasses import dataclass
from pathlib import Path
from itertools import combinations
from math import exp, factorial
from typing import Callable
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
SEED, MAX_SCORE, RUNS = 20260704, 8, 5000

@dataclass(frozen=True)
class Fixture:
    home: str
    away: str

@dataclass(frozen=True)
class MatchDistribution:
    fixture: Fixture
    matrix: np.ndarray
    expected_home_goals: float
    expected_away_goals: float

@dataclass(frozen=True)
class MatchResult:
    fixture: Fixture
    home_goals: int
    away_goals: int

def capped_poisson(rate: float, maximum: int = MAX_SCORE) -> np.ndarray:
    exact = np.array([exp(-rate) * rate ** k / factorial(k) for k in range(maximum)])
    return np.append(exact, max(0.0, 1.0 - exact.sum()))

def score_matrix(home_rate: float, away_rate: float) -> np.ndarray:
    matrix = np.outer(capped_poisson(home_rate), capped_poisson(away_rate))
    return matrix / matrix.sum()

def sample_match(distribution: MatchDistribution, rng: np.random.Generator) -> MatchResult:
    index = int(rng.choice(distribution.matrix.size, p=distribution.matrix.ravel()))
    home_goals, away_goals = np.unravel_index(index, distribution.matrix.shape)
    return MatchResult(distribution.fixture, int(home_goals), int(away_goals))

def simulate_match(distribution: MatchDistribution, runs: int, seed: int) -> tuple[pd.DataFrame, pd.Series]:
    rng = np.random.default_rng(seed)
    results = [sample_match(distribution, rng) for _ in range(runs)]
    scores = pd.Series([(r.home_goals, r.away_goals) for r in results]).value_counts(normalize=True).rename('frequency')
    outcomes = pd.Series(['H' if r.home_goals > r.away_goals else 'A' if r.home_goals < r.away_goals else 'D' for r in results]).value_counts(normalize=True).reindex(['H', 'D', 'A'], fill_value=0.0)
    return scores.rename_axis('score').reset_index(), outcomes

def round_robin(teams: list[str]) -> list[Fixture]:
    if len(teams) != len(set(teams)) or len(teams) < 2:
        raise ValueError('At least two unique teams are required')
    return [Fixture(a, b) for a, b in combinations(teams, 2)]

def standings(teams: list[str], results: list[MatchResult]) -> pd.DataFrame:
    table = {team: {'played': 0, 'won': 0, 'drawn': 0, 'lost': 0, 'gf': 0, 'ga': 0, 'points': 0} for team in teams}
    for result in results:
        h, a, hg, ag = result.fixture.home, result.fixture.away, result.home_goals, result.away_goals
        table[h]['played'] += 1; table[a]['played'] += 1
        table[h]['gf'] += hg; table[h]['ga'] += ag; table[a]['gf'] += ag; table[a]['ga'] += hg
        if hg > ag:
            table[h]['won'] += 1; table[a]['lost'] += 1; table[h]['points'] += 3
        elif hg < ag:
            table[a]['won'] += 1; table[h]['lost'] += 1; table[a]['points'] += 3
        else:
            table[h]['drawn'] += 1; table[a]['drawn'] += 1; table[h]['points'] += 1; table[a]['points'] += 1
    frame = pd.DataFrame.from_dict(table, orient='index').rename_axis('team').reset_index()
    frame['goal_difference'] = frame['gf'] - frame['ga']
    frame = frame.sort_values(['points', 'goal_difference', 'gf', 'team'], ascending=[False, False, False, True], kind='stable').reset_index(drop=True)
    frame.insert(0, 'position', np.arange(1, len(frame) + 1))
    return frame

def simulate_group(teams: list[str], distribution_for: Callable[[Fixture], MatchDistribution], runs: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    fixtures = round_robin(teams)
    distributions = [distribution_for(f) for f in fixtures]
    counts = pd.DataFrame(0, index=teams, columns=range(1, len(teams) + 1), dtype=int)
    for _ in range(runs):
        table = standings(teams, [sample_match(d, rng) for d in distributions])
        for row in table.itertuples():
            counts.loc[row.team, row.position] += 1
    probabilities = counts / runs
    probabilities.columns = [f'finish_{c}' for c in probabilities.columns]
    probabilities['qualify_top_2'] = probabilities['finish_1'] + probabilities['finish_2']
    return probabilities.sort_values(['qualify_top_2', 'finish_1'], ascending=False)


In [2]:
# Fit the Phase 5 baseline using training-period data only, then build neutral-site group fixtures.
matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id'])
model_data = matches.loc[matches['date'] >= '2000-01-01'].reset_index(drop=True)
split = int(len(model_data) * 0.8)
train = model_data.iloc[:split]
X = np.column_stack([train['elo_difference'].to_numpy() / 400.0, train['neutral'].astype(float)])
home_model = PoissonRegressor(alpha=0.1, max_iter=1000).fit(X, train['home_score'])
away_model = PoissonRegressor(alpha=0.1, max_iter=1000).fit(X, train['away_score'])
latest_ratings = pd.concat([
    matches[['date', 'home_team', 'home_elo']].rename(columns={'home_team': 'team', 'home_elo': 'rating'}),
    matches[['date', 'away_team', 'away_elo']].rename(columns={'away_team': 'team', 'away_elo': 'rating'})
]).sort_values('date').groupby('team').tail(1).set_index('team')['rating']

teams = ['Argentina', 'England', 'France', 'Spain']
def distribution_for(fixture: Fixture) -> MatchDistribution:
    elo_difference = float(latest_ratings[fixture.home] - latest_ratings[fixture.away])
    features = np.array([[elo_difference / 400.0, 1.0]])
    home_rate = float(home_model.predict(features)[0])
    away_rate = float(away_model.predict(features)[0])
    return MatchDistribution(fixture, score_matrix(home_rate, away_rate), home_rate, away_rate)

example_distribution = distribution_for(Fixture('Argentina', 'England'))
score_frequencies, outcome_frequencies = simulate_match(example_distribution, RUNS, SEED)
group_probabilities = simulate_group(teams, distribution_for, RUNS, SEED)
print('Example match expected goals:', example_distribution.expected_home_goals, example_distribution.expected_away_goals)
print('Example outcome frequencies:')
print(outcome_frequencies)
group_probabilities

C:\Users\moksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\moksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Example match expected goals: 1.8961691801882476 1.0010298754984932
Example outcome frequencies:
H    0.5848
D    0.2190
A    0.1962
Name: proportion, dtype: float64


,finish_1,finish_2,finish_3,finish_4,qualify_top_2
Argentina,0.4630,0.2668,0.171,0.0992,0.7298
Spain,0.2088,0.2688,0.269,0.2534,0.4776
France,0.1862,0.2620,0.273,0.2788,0.4482
England,0.1420,0.2024,0.287,0.3686,0.3444


In [3]:
# Embedded validation checks: seed repeatability, arithmetic, tie-breaks, totals, and fixture coverage.
fixtures = round_robin(teams)
assert len(fixtures) == 6 and len({(f.home, f.away) for f in fixtures}) == 6
first = [sample_match(example_distribution, np.random.default_rng(7)) for _ in range(1)]
second = [sample_match(example_distribution, np.random.default_rng(7)) for _ in range(1)]
assert first == second

toy_teams = ['Alpha', 'Beta', 'Gamma']
toy_results = [
    MatchResult(Fixture('Alpha', 'Beta'), 2, 0),
    MatchResult(Fixture('Alpha', 'Gamma'), 1, 1),
    MatchResult(Fixture('Beta', 'Gamma'), 3, 0),
]
toy = standings(toy_teams, toy_results).set_index('team')
assert toy.loc['Alpha', 'points'] == 4 and toy.loc['Alpha', 'goal_difference'] == 2
assert toy.loc['Beta', 'points'] == 3 and toy.loc['Beta', 'gf'] == 3
assert toy.loc['Gamma', 'points'] == 1
blank = standings(['Zulu', 'Alpha'], [])
assert blank['team'].tolist() == ['Alpha', 'Zulu']  # documented unresolved-tie fallback
assert np.allclose(group_probabilities.filter(like='finish_').sum(axis=1), 1.0)
assert np.isclose(group_probabilities['finish_1'].sum(), 1.0)
assert np.isclose(group_probabilities['qualify_top_2'].sum(), 2.0)
repeat_group = simulate_group(teams, distribution_for, RUNS, SEED)
pd.testing.assert_frame_equal(group_probabilities, repeat_group)
assert np.isclose(outcome_frequencies.sum(), 1.0)
print('All Phase 6 validation checks passed.')

headers = ['team', *group_probabilities.columns.tolist()]
table_lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
for team, row in group_probabilities.iterrows():
    table_lines.append('| ' + ' | '.join([team, *[f'{value:.4f}' for value in row]]) + ' |')
probability_table = '\n'.join(table_lines)

report = f'''# Phase 6 Simulation Report

Generated by `notebooks/04_simulation_baseline.ipynb` with seed {SEED} and {RUNS:,} Monte Carlo runs.

## Configuration

- Teams: {', '.join(teams)}
- Format: single round robin ({len(fixtures)} matches), neutral venue
- Score model: separate regularized Poisson regressions using chronological pre-match Elo features
- Display/sampling cap: 8 means 8 or more goals
- Tie-break order: points, goal difference, goals scored, alphabetical team name
- Qualification: top two positions

## Qualification and finishing probabilities

{probability_table}

## Example match: Argentina vs England

- Expected goals: {example_distribution.expected_home_goals:.3f}–{example_distribution.expected_away_goals:.3f}
- Simulated outcomes: home {outcome_frequencies['H']:.3f}, draw {outcome_frequencies['D']:.3f}, away {outcome_frequencies['A']:.3f}

## Limitations

The model assumes independent goal counts and uses only Elo difference and venue. The alphabetical fallback is deterministic but is not a substitute for tournament-specific head-to-head, fair-play, or drawing-lots rules. Monte Carlo estimates contain sampling error.
'''
report_path = ROOT / 'reports' / 'phase6_simulation_report.md'
report_path.write_text(report, encoding='utf-8')
print('Wrote', report_path)

All Phase 6 validation checks passed.
Wrote G:\Coding\nations-ai\reports\phase6_simulation_report.md
